In [14]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul3-train/2_3_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
print(x_train.shape, y_train.shape)

test_data = np.load('Numarul3-test/2_3_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
print(x_test.shape, y_test.shape)

N_CLASSES = len(np.unique(y_train))
print(f"N_CLASSES: {N_CLASSES}")

print(f"shape-ul imaginilor este de forma: {x_train[0].shape}")
print(f"dtype-ul imaginilor este de forma: {x_train[0].dtype}")
print(f"dtype-ul labelurilor este de forma: {y_train[0].dtype}")

H = [img.shape[0] for img in x_train]
W = [img.shape[1] for img in x_train]
print(min(H), max(H))
print(min(W), max(W))

# 32 MAI MARE DECAT 0.5 * MAXIM
H = W = 48


(25000,) (25000,)
(5000,) (5000,)
N_CLASSES: 5
shape-ul imaginilor este de forma: (55, 23, 3)
dtype-ul imaginilor este de forma: uint8
dtype-ul labelurilor este de forma: int64
20 60
20 60


In [15]:

class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.images = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx])
        label = self.labels[idx]

        image = image.resize(size=(H, H))
        image = torch.as_tensor(data=np.array(image), dtype=torch.float32)
        image = image / 255.0
        image = torch.permute(input=image, dims=(2, 0, 1))
        label = torch.as_tensor(data=label, dtype=torch.int64)

        return image, label


train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
class Model(nn.Module):
    def __init__(self, in_channels=3, n_classes=N_CLASSES, act=nn.functional.relu):
        super().__init__()

        self.act = act

        # IN -> (3, 48 , 48)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=48, kernel_size=3, padding=1) # -> (48, 48, 48)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=3) # -> (48, 16, 16)

        self.conv2 = nn.Conv2d(in_channels=48, out_channels=32, kernel_size=3, padding=1) # -> (32, 16, 16)
        self.pool2 = nn.MaxPool2d(kernel_size=2) # -> (32, 8, 8)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1) # -> (32, 8, 8)
        self.pool3 = nn.MaxPool2d(kernel_size=2) # -> (32, 4, 4)

        self.fc1 = nn.Linear(in_features=32 * 4 * 4, out_features=512)
        self.drop1 = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=512, out_features=256)
        self.drop2 = nn.Dropout(p=0.3)
        self.fc3 = nn.Linear(in_features=256, out_features=n_classes)

    def forward(self, x):
        x = self.pool1(self.act(self.conv1(x)))
        x = self.pool2(self.act(self.conv2(x)))
        x = self.pool3(self.act(self.conv3(x)))

        x = x.flatten(1)
        x = self.drop1(self.act(self.fc1(x)))
        x = self.drop2(self.act(self.fc2(x)))
        x = self.fc3(x)

        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 10


def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total


def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total

best = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"Epoch {epoch}: train loss={train_loss:.4f}, acc={train_acc:.4f} | test loss={test_loss:.4f}, acc={test_acc:.4f}")
    if best < test_acc:
        best = test_acc  
        torch.save(model, 'model.pt')

torch.save(model, 'model.pt')


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')

Epoch 0: train loss=1.2311, acc=0.4718 | test loss=0.9992, acc=0.5924
Epoch 1: train loss=0.9537, acc=0.6208 | test loss=0.8624, acc=0.6550
Epoch 2: train loss=0.8429, acc=0.6688 | test loss=0.8019, acc=0.6858
Epoch 3: train loss=0.7747, acc=0.7022 | test loss=0.8247, acc=0.6812
Epoch 4: train loss=0.7176, acc=0.7238 | test loss=0.7201, acc=0.7242
Epoch 5: train loss=0.6640, acc=0.7465 | test loss=0.6915, acc=0.7416
Epoch 6: train loss=0.6229, acc=0.7636 | test loss=0.7324, acc=0.7232
Epoch 7: train loss=0.5943, acc=0.7763 | test loss=0.6794, acc=0.7416
Epoch 8: train loss=0.5526, acc=0.7954 | test loss=0.7180, acc=0.7404
Epoch 9: train loss=0.5172, acc=0.8076 | test loss=0.6506, acc=0.7612
0.7612
